In [2]:
#| default_exp text

In [3]:
#| hide
from nbdev.showdoc import *

In [4]:
#| export
import re
from fastcore.all import L, Path
from litesearch.sanskrit import DANDA, DDANDA, CITE_RE, cite_parts, DEVANAGARI

## Where a verse ends

Three markers, tried in order of how much they say:

1. a citation, `// Mn_1.1 //`, which is also a stable id
2. a Devanagari verse number, `॥ १॥`
3. a daṇḍa at end of line

A cut swallows the `>` gloss lines trailing it, so a translation stays with its verse.

In [5]:
#| export
# A Devanagari verse number: `॥ १॥` or `॥ 12 ॥`
_DEVNUM = re.compile(r'॥\s*([\d०-९]+)\s*॥')

_GLOSS = re.compile(r'(?:[ \t]*\n)?[ \t]*>[^\n]*')
_HEAD  = re.compile(r'^[ \t]*#{1,6}[ \t]+\S', re.M)
_LINENUM = re.compile(r'^[ \t]*(\d+)(?:[ \t]*(?=\n|$)|(?=[^\W\d_]))', re.M)

def _with_gloss(text:str, end:int) -> int:
    'Extend a cut past the gloss lines that trail it.'
    while (m := _GLOSS.match(text, end)) and m.end() > end: end = m.end()
    return end

def verse_spans(text:str) -> L:
    'Split `text` into verse units, returning `(start, end, text, citation)`.'
    if not (text or '').strip(): return L()
    cuts = []
    for m in CITE_RE.finditer(text): cuts.append((_with_gloss(text, m.end()), f'{m.group(1)}_{m.group(2)}'))
    if not cuts:
        for m in _DEVNUM.finditer(text): cuts.append((_with_gloss(text, m.end()), m.group(1)))
    if not cuts:
        for m in re.finditer(r'(?:॥|\|\||//)[ \t]*(?=\n|$)', text):
            cuts.append((_with_gloss(text, m.end()), None))
    if not cuts and len(nums := list(_LINENUM.finditer(text))) > 1:
        out = L()
        for i, m in enumerate(nums):
            end = nums[i + 1].start() if i + 1 < len(nums) else len(text)
            if (seg := text[m.end():end]).strip(): out.append((m.end(), end, seg.strip(), m.group(1)))
        return out
    cuts += [(m.start(), None) for m in _HEAD.finditer(text) if m.start() > 0]
    cuts.sort()
    out, prev = L(), 0
    for end, cite in cuts:
        if end <= prev: continue
        if (seg := text[prev:end]).strip(): out.append((prev, end, seg.strip(), cite))
        prev = end
    if (tail := text[prev:]).strip(): out.append((prev, len(text), tail.strip(), None))
    return out

_CLAUSE = re.compile(r'(?:॥|।|//|/|\|)[ \t]*')

def _clause_spans(seg:str) -> L:
    'Split an over-long unit at its clause daṇḍas, never mid-clause.'
    parts, prev = L(), 0
    for m in _CLAUSE.finditer(seg):
        if (s := seg[prev:m.end()]).strip(): parts.append(s.strip())
        prev = m.end()
    if (t := seg[prev:]).strip(): parts.append(t.strip())
    return parts or L([seg.strip()])

def _atoms(seg:str, max_chars:int) -> L:
    'The smallest pieces an over-long unit may be cut into, each with its gloss attached.'
    out = L()
    for p in _clause_spans(seg):
        if len(p) <= max_chars: out.append(p); continue
        lines = [l.strip() for l in p.splitlines() if l.strip()]
        out += (lines if len(lines) > 1 else [p])
    atoms = L()
    for p in out:
        if p.lstrip().startswith('>') and atoms: atoms[-1] += '\n' + p
        else: atoms.append(p)
    return atoms

In [6]:
from ganapati.text import cite_parts, verse_spans

assert cite_parts('BrhUp_1,1.2') == ('BrhUp', ['1','1','2'])
assert cite_parts('Mn_1.1') == ('Mn', ['1','1'])
assert cite_parts('IsUp_4[57M]') == ('IsUp', ['4'])      # the alternate-edition number is dropped
assert cite_parts('not a citation') == ('not a citation', [])

two = ('īśā vāsyam idaṃ sarvaṃ | tena tyaktena bhuñjīthāḥ || IsUp_1 ||\n'
       'kurvann eveha karmāṇi | evaṃ tvayi nānyatheto || IsUp_2 ||\n')
sp = verse_spans(two)
assert len(sp) == 2
assert [c for *_, c in sp] == ['IsUp_1', 'IsUp_2']
assert sp[0][2].startswith('īśā') and sp[0][2].endswith('||')

numbered = ('0nārāyaṇaṃ namaskṛtya\n'
            '  1\n'
            'lomaharṣaṇaputra ugraśravāḥ\n'
            '2samāsīnān abhyagacchat')
sp = verse_spans(numbered)
assert len(sp) == 3
assert [s for *_, s, _ in sp] == [
    'nārāyaṇaṃ namaskṛtya', 'lomaharṣaṇaputra ugraśravāḥ', 'samāsīnān abhyagacchat']

## Chunkers

`VerseChunker` is chonkie-shaped and drops into `add_doc(chunker=...)`. It cuts on verse
boundaries, packs short verses towards `target`, and splits long prose at `max_chars`.
`ProseChunker` is the same with a bigger budget and `pack_cited=True`.

In [7]:
#| export
VERSE_TARGET, VERSE_MAX = 300, 900

def _width(parts) -> int:
    'Length the parts will have once joined — the newlines count against the budget too.'
    return sum(map(len, parts)) + max(len(parts) - 1, 0)

def chunk_verses(text:str,                 # source text
                 target:int=VERSE_TARGET,  # pack short units forward up to this many chars
                 max_chars:int=VERSE_MAX,  # split a unit longer than this at clause boundaries
                 pack_cited:bool=False,    # let consecutive cited units share a chunk up to `target`
                 ) -> L:
    'Verse-aware chunks: cut on citation/daṇḍa, pack short units, split long prose.'
    units = verse_spans(text)
    if not units: return L([text.strip()]) if (text or '').strip() else L()
    out, buf, cited = L(), [], False
    def flush():
        nonlocal buf, cited
        if buf: out.append('\n'.join(buf).strip())
        buf, cited = [], False
    for _, _, seg, cite in units:
        if _HEAD.match(seg): flush()
        if len(seg) > max_chars:
            flush()
            cur = []
            for a in _atoms(seg, max_chars):
                if cur and _width(cur) + len(a) > max_chars: out.append('\n'.join(cur)); cur = []
                cur.append(a)
            if cur: out.append('\n'.join(cur))
            continue
        if buf and _width(buf) + len(seg) > max_chars: flush()
        buf.append(seg)
        if cite: cited = True
        if (cited and not pack_cited) or _width(buf) >= target: flush()
    flush()
    return out.filter(lambda s: s.strip())

class _Chunk:
    'Minimal chonkie-compatible chunk (chonkie.types.Chunk is imported lazily to keep this light).'
    def __init__(self, text, start_index, end_index, token_count=0):
        self.text, self.start_index, self.end_index, self.token_count = text, start_index, end_index, token_count
    def __repr__(self): return f'_Chunk({self.text[:40]!r}…)'

def _mk_chunks(texts, src):
    'Wrap chunk strings as chonkie chunks, carrying real offsets into `src` where they can be found.'
    try: from chonkie.types import Chunk
    except Exception: Chunk = _Chunk
    out, pos = [], 0
    for t in texts:
        i = src.find(t[:60], pos) if t else -1
        st = i if i >= 0 else pos
        out.append(Chunk(text=t, start_index=st, end_index=st+len(t), token_count=0))
        pos = st + len(t)
    return out

class VerseChunker:
    'A chonkie-shaped chunker whose cuts land on verse boundaries.'
    def __init__(self, target:int=VERSE_TARGET, max_chars:int=VERSE_MAX, pack_cited:bool=False):
        self.target, self.max_chars, self.chunk_size = target, max_chars, max_chars
        self.pack_cited = pack_cited
    def chunk(self, text:str):
        return _mk_chunks(chunk_verses(text, self.target, self.max_chars, self.pack_cited), text or '')
    def __call__(self, text:str): return self.chunk(text)
    def __repr__(self):
        return f'{type(self).__name__}(target={self.target}, max_chars={self.max_chars}, pack_cited={self.pack_cited})'

class ProseChunker(VerseChunker):
    '`VerseChunker` tuned for Sanskrit prose — commentary, bhāṣya, the prose Upaniṣads.'
    def __init__(self, target:int=700, max_chars:int=1400, pack_cited:bool=True):
        super().__init__(target, max_chars, pack_cited)

In [8]:
cs = VerseChunker().chunk(two)
assert len(cs) == 2
assert two[cs[0].start_index:cs[0].end_index] == cs[0].text   # offsets point back at the source

# ProseChunker packs cited units instead, which is what commentary wants
assert len(ProseChunker().chunk(two)) == 1

# with no citation to cut on, the budget decides. 3,040 characters of daṇḍa-separated prose:
long = ' | '.join(['padam yatra tatra sarvatra gacchati'] * 80) + ' ||'
assert len(chunk_verses(long, 300, 900)) == 4
assert len(chunk_verses(long, 700, 1400)) == 3

# litesearch packs across segments only for a chunker that asks; the verse profile does not,
# because a reader that emits one verse per page is already the right size
from litesearch.tree import _pack_target
assert _pack_target(VerseChunker()) == 0
assert _pack_target(ProseChunker()) == 700

## Four formats, one shape

Every reader returns `(pages, meta)`. `sanskrit_parse` picks between them from the first 4 KB.

| reader | source |
|----|----|
| `gretil_parse` | GRETIL plain text, `*_u.htm` |
| `tei_parse` | TEI XML |
| `vr_xml_parse` | Vedic Reserve `<lyrics>` XML |
| `dcs_parse` | DCS CoNLL-U, which carries lemmas |

In [9]:
#| export
_GRETIL_END = re.compile(r'COPYRIGHT AND TERMS OF USAGE AS FOR SOURCE FILE\.|'
                         r'For further information see:\s*\n\s*http\S+', re.I)

def _detag(t:str) -> str:
    import html
    t = re.sub(r'<(script|style)[^>]*>.*?</\1>', ' ', t, flags=re.S | re.I)
    return html.unescape(re.sub(r'<[^>]+>', '', t)).replace('\xa0', ' ')

def gretil_parse(src) -> tuple:
    'GRETIL plain-text (`*_u.htm`) to `(pages, meta)`.'
    txt = _detag(Path(src).read_text(encoding='utf-8', errors='replace') if _isfile(src) else str(src))
    head, body = '', txt
    if (m := _GRETIL_END.search(txt)):
        head, body = txt[:m.end()], txt[m.end():]
        # the diacritics table sits *after* the banner; drop lines up to the last of its entries
        if (d := list(re.finditer(r'^\s*\w[\w\s]*\s{2,}[^\x00-\x7f]\s*$', body, re.M))):
            head, body = head + body[:d[-1].end()], body[d[-1].end():]
    body = re.sub(r'\n{3,}', '\n\n', body).strip()
    blocks = [b for b in re.split(r'\n\s*\n', body) if b.strip()]
    while blocks and not _has_sanskrit(blocks[0]):
        head += '\n\n' + blocks.pop(0)
    body = '\n\n'.join(blocks)
    spans = verse_spans(body)
    pages = [(i, s) for i, (_, _, s, _) in enumerate(spans)] if len(spans) > 1 else list(enumerate(blocks))
    ttl = _first_line(head) or (Path(src).stem if _isfile(src) else 'gretil')
    return pages or [(0, body)], dict(fmt='gretil', title=ttl, header=head.strip()[:2000])

def tei_parse(src) -> tuple:
    'GRETIL `corpustei` / SARIT TEI to `(pages, meta)`.'
    from xml.etree import ElementTree as ET
    raw = Path(src).read_text(encoding='utf-8', errors='replace') if _isfile(src) else str(src)
    root = ET.fromstring(raw)
    ns = {'t': 'http://www.tei-c.org/ns/1.0'}
    def tag(e): return e.tag.split('}')[-1]
    def txt(e):
        parts = []
        for n in e.iter():
            if tag(n) in ('note', 'app', 'rdg', 'lem'): continue
            if n.text: parts.append(n.text)
            if n is not e and n.tail: parts.append(n.tail)
        return re.sub(r'\s+', ' ', ' '.join(parts)).strip()
    body = root.find('.//t:text/t:body', ns) or root.find('.//text/body') or root
    pages, path = [], []
    def walk(el, depth=0):
        for ch in el:
            t = tag(ch)
            if t == 'div':
                nm = ch.get('type') or 'div'; n = ch.get('n') or ''
                path.append(f"{nm} {n}".strip())
                pages.append((len(pages), f"{'#' * min(depth + 2, 6)} {path[-1]}"))
                walk(ch, depth + 1); path.pop()
            elif t == 'lg':
                cid = ch.get('{http://www.w3.org/XML/1998/namespace}id') or ch.get('id') or ''
                # `<l>` often already carries a trailing `//`; appending the id beside it would
                # leave `// || Manu_1.2 ||` and two boundary markers where there is one boundary.
                s = re.sub(r'\s*(?://|\|\|)\s*$', '', txt(ch))
                if s: pages.append((len(pages), f"{s} || {cid} ||" if cid else s))
            elif t in ('p', 'ab'):
                if (s := txt(ch)): pages.append((len(pages), s))
            elif t in ('note', 'app'): continue
            else: walk(ch, depth)
    walk(body)
    ttl = (root.findtext('.//t:titleStmt/t:title', default='', namespaces=ns)
           or root.findtext('.//titleStmt/title', default='') or '').strip()
    return _merge_pages(pages), dict(fmt='tei', title=ttl or (Path(src).stem if _isfile(src) else 'tei'))

def vr_xml_parse(src) -> tuple:
    'vedicreader `<lyrics>` XML to `(pages, meta)`.'
    from xml.etree import ElementTree as ET
    raw = Path(src).read_text(encoding='utf-8', errors='replace') if _isfile(src) else str(src)
    root = ET.fromstring(raw)
    pages, ttl = [], ''
    for so, s in enumerate(root.iter('section')):
        nm = s.get('name') or f'section {so}'
        lines = [l for l in s.iterfind('line')
                 if (l.text or '').strip() and (l.get('exclude_from_display') or 'false').lower() != 'true']
        if nm == 'title' and lines and not ttl: ttl = (lines[0].text or '').strip()
        if not lines: continue
        pages.append((len(pages), f'## {nm}'))
        buf = []
        for l in lines:
            buf.append((l.text or '').strip())
            if (g := (l.get('meaning') or l.get('caption') or '').strip()): buf.append('> ' + g)
        body = '\n'.join(buf)
        if not re.search(r'[।॥]\s*$', body.split('\n>')[0]): body += '\n॥'
        if (sm := (s.get('meaning') or '').strip()): body += '\n> ' + sm
        pages.append((len(pages), body))
    cat = (root.findtext('category') or '').strip()
    tags = (root.findtext('tags') or '').strip()
    return _merge_pages(pages), dict(fmt='vedicreader', title=ttl, category=cat, tags=tags)

def dcs_parse(src) -> tuple:
    'DCS / ambuda analysed text to `(pages, meta)`.'
    raw = Path(src).read_text(encoding='utf-8', errors='replace') if _isfile(src) else str(src)
    pages, cur, lem, cid = [], [], [], None
    def flush():
        if not cur: return
        v = ' '.join(cur)
        # lemmas on the verse's own page, so they travel with it rather than opening the next unit
        pages.append((len(pages), (f"{v} || {cid} ||" if cid else v)
                      + ('\n> lemmas: ' + ' '.join(dict.fromkeys(lem)) if lem else '')))
    for ln in raw.splitlines():
        if ln.startswith('#'):
            if (m := re.search(r'id\s*=\s*(\S+)', ln)): flush(); cur, lem, cid = [], [], m.group(1)
            continue
        if not ln.strip():
            flush(); cur, lem, cid = [], [], None
            continue
        f = ln.split('\t')
        if f and f[0].strip(): cur.append(f[0].strip())
        if len(f) > 1 and f[1].strip(): lem.append(f[1].strip())
    flush()
    return _merge_pages(pages), dict(fmt='dcs', title=(Path(src).stem if _isfile(src) else 'dcs'))

def _isfile(x):
    try: return Path(x).is_file()
    except (OSError, ValueError): return False

def _first_line(t):
    for l in (t or '').splitlines():
        if (s := l.strip()) and not s.startswith('http') and len(s) > 3: return s[:120]
    return ''

def _merge_pages(pages, per:int=1):
    'One page per structural unit is fine for the tree; collapse to text blocks for build_tree.'
    return [(i, t) for i, t in pages if (t or '').strip()]

_IAST_DIAC = re.compile('[āīūṛṝḷḹṅñṭḍṇśṣṃḥĀĪŪṚṄÑṬḌṆŚṢṂḤ]')

def _has_sanskrit(block:str) -> bool:
    'Whether a block carries any Sanskrit signal — script, diacritic, daṇḍa or citation.'
    return bool(DEVANAGARI.search(block) or _IAST_DIAC.search(block)
                or DANDA in block or DDANDA in block or CITE_RE.search(block))

_SANSKRIT_EXTS = '.xml,.tei,.htm,.html,.conllu,.txt'

_DANDA_EOL = re.compile(r'(?:[।॥]|//|/|\|\||\|)[ \t]*$', re.M)

def is_sanskrit(text:str, thresh:float=0.15) -> bool:
    'Whether `text` reads as Sanskrit — by script, by transliteration, or by structure.'
    s = (text or '')[:20000]
    if not s.strip(): return False
    if len(DEVANAGARI.findall(s)) / max(len(s), 1) > thresh: return True
    lines = max(len(s.splitlines()), 1)
    strict = len(CITE_RE.findall(s)) + s.count(DDANDA)
    if strict >= 3 and strict / lines > 0.05: return True
    loose = strict + len(_DANDA_EOL.findall(s))
    return (len(_IAST_DIAC.findall(s)) / max(len(s), 1) > 0.01) and loose >= 3 and loose / lines > 0.02

def sanskrit_parse(src) -> tuple:
    'Parse any supported Sanskrit source to `(pages, meta)`, picking the reader by shape.'
    p = Path(src) if _isfile(src) else None
    raw = p.read_text(encoding='utf-8', errors='replace')[:4000] if p else str(src)[:4000]
    sfx = p.suffix.lower() if p else ''
    if sfx in ('.conllu',) or re.search(r'^#\s*id\s*=', raw, re.M): return dcs_parse(src)
    if '<lyrics' in raw: return vr_xml_parse(src)
    if 'tei-c.org' in raw or '<TEI' in raw or '<teiHeader' in raw: return tei_parse(src)
    if sfx in ('.htm', '.html') or 'GRETIL' in raw: return gretil_parse(src)
    if sfx == '.xml':
        return vr_xml_parse(src) if '<lyrics' in raw else tei_parse(src)
    return gretil_parse(src)

In [10]:
assert is_sanskrit('śrīmātā śrīmahārājñī || LSN_1 ||\nśrīmatsiṃhāsaneśvarī || LSN_2 ||\n'
                   'cidagnikuṇḍasambhūtā || LSN_3 ||')
assert is_sanskrit('अथ योगानुशासनम् ॥ १॥')
assert not is_sanskrit('This is an ordinary English paragraph about nothing in particular.')
assert not is_sanskrit('')
assert _detag('dharma&nbsp;artha') == 'dharma artha'

In [13]:
sanskrit_parse('mahabharata.htm')[0][0]

(0,
 'nārāyaṇaṃ namaskṛtya naraṃ caiva narottamam\ndevīṃ sarasvatīṃ caiva tato jayam udīrayet')

In [14]:
#| hide
import nbdev; nbdev.nbdev_export()